In [9]:
import numpy as np
from clarautils import CCol, TableFields, QueryableTable

## 1. The declaration IS the table type

A table is structured by a `TableFields` declaration: each field is declared **exactly once** as a `Range | Item` union.

- **Range** — the column container. `CCol` (=`ConstraintColumn`) is the queryable column type of `QueryableTable`.
- **Item** — the scalar cell type (a concrete numpy dtype).

From this one declaration the framework derives everything else: the structured-array dtype, the `item_type` (a typed `NamedTuple` for rows) and the `range_type` (one column per field for slice selections). Both are generated internally and cached per declaration — nothing is derived by hand.

A field whose item member is `np.object_` is a **reference column**: it rides along as an object field, so arbitrary Python values (here `np.dtype` instances) can live in the same packed table.

In [10]:
class DTableFields(TableFields):
    signed: CCol | np.bool_
    abs_min: CCol | np.uint64
    max: CCol | np.uint64
    bits: CCol | np.uint8
    type: CCol | np.object_

## 2. Build the data

One row per integer dtype: whether it is signed, how far the negative side reaches (`abs_min = -iinfo.min`, so -128 fits int8), the positive bound `max`, the `bits`, and the `np.dtype` instance itself in the object column.

Plain tuples in, no adapters or creator classes needed.

In [11]:
def build_type_tbl():
    kind = {'u': False, 'i': True}
    sizes = [1, 2, 4, 8]
    types = np.array([
        np.dtype(f"{k}{s}")
        for k in kind
        for s in sizes
    ])

    return [(
        kind[t.kind],
        -np.iinfo(t).min,
        np.iinfo(t).max,
        np.iinfo(t).bits,
        t
    ) for t in types]

data = build_type_tbl()
print(data)

[(False, 0, 255, 8, dtype('uint8')), (False, 0, 65535, 16, dtype('uint16')), (False, 0, 4294967295, 32, dtype('uint32')), (False, 0, 18446744073709551615, 64, dtype('uint64')), (True, 128, 127, 8, dtype('int8')), (True, 32768, 32767, 16, dtype('int16')), (True, 2147483648, 2147483647, 32, dtype('int32')), (True, 9223372036854775808, 9223372036854775807, 64, dtype('int64'))]


## 3. Stick it together

`QueryableTable(data, DTableFields)` — the declaration alone drives the construction:

- columns are built and attached as attributes (validated against the declared ranges at build time),
- `tbl[i]` will return the item variant, `tbl[i:j]` / `tbl[indices]` the range variant,
- `print(qtbl)` shows the table length — repr deliberately stays cheap.

In [12]:
qtbl = QueryableTable(data, DTableFields)
print(qtbl)

QueryableTable(len=8)


## 4. Columns are lazy queryable CCols

Each field attribute holds its own `CCol`. A `CCol` wraps the raw numpy column; comparisons (`==`, `>=`, `<`, ...) do **not** evaluate — they build lazy `Constraint` objects, and `&` / `|` compose them into a `Query`. Nothing touches the data until `.indices`, `.get_first` or `.get_all` is called.

In [13]:
print(qtbl.signed)

CCol(signed): array([False, False, False, False,  True,  True,  True,  True])


## 5. Access a row

`qtbl[0]` returns the **item variant** — a typed `NamedTuple` derived from the
declaration, one scalar per field. An `int` or `np.integer` key selects a row;
anything else (slice, index array, boolean mask) selects the **range variant**
instead: one column object per field over the selection.

In [14]:
row = qtbl[0]
print(row)

DTable(signed=np.False_, abs_min=np.uint64(0), max=np.uint64(255), bits=np.uint8(8), type=dtype('uint8'))


## 6. Rows are typed — no workarounds

The row is a real `NamedTuple` with its own field types, so attribute access just works — with full IDE/linting support, no `rec[0]` positional juggling, no `.view()` gymnastics.

In [15]:
if row.signed:
    print("Is signed")
else:
    print("Not signed")

Not signed


## 7. Actual querying

`get_first` on a column returns the first matching cell of that column. Here: the smallest signed dtype whose `max` exceeds int16's max — which is `int32`.

Notes on the query machinery:

- `(qtbl.signed == True) & (...)` — comparisons build `Constraint` leaves, `&` / `|` compose a `Query` tree; evaluation happens once, in `.indices`, and the result is an index array (`np.intersect1d` / `np.union1d` at the top).
- The query composes lazily, so it can be built up, stored and reused before it ever runs.
- `col == Undefined` builds an always-true constraint — handy for parameterized queries that can switch a condition off.
- Object column caveat: compare against `np.dtype(...)` **instances**, not against the bare type class — `object_array == np.uint32` crashes inside numpy.

In [16]:
item = qtbl.type.get_first((qtbl.signed == True) & (qtbl.max > np.iinfo(np.int16).max))
print(item)

int32
